# Módulo 6 · Clase 3 — Probabilidad

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Probabilidad: medir la incertidumbre |
| 2 | Eventos, espacio muestral, axiomas |
| 3 | Probabilidad condicional |
| 4 | Teorema de Bayes — el corazón de la IA probabilística |
| 5 | Distribuciones discretas: uniforme y binomial |
| 6 | Distribución normal — la más importante en estadística |
| 7 | Otras distribuciones útiles en IA |

In [ ]:
import subprocess, sys
for pkg in ['numpy', 'matplotlib', 'scipy']:
    try: __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
print("Librerías listas")

---
## 1. ¿Qué es la probabilidad?

```
La probabilidad mide el grado de incertidumbre sobre un evento.

P(A) ∈ [0, 1]
  P = 0  → imposible
  P = 1  → certeza
  P = 0.5 → igualmente probable que ocurra o no

¿Por qué importa en IA?

  ChatGPT no da la respuesta con certeza, genera la palabra
  MÁS PROBABLE dada la secuencia anterior.

  Un clasificador de spam no dice "es spam" sino
  P(spam | email) = 0.97.

  Un modelo de difusión (DALL-E, Stable Diffusion) genera
  imágenes muestreando de una distribución de probabilidad.

Dos interpretaciones:
  Frecuentista: P = frecuencia en el largo plazo
  Bayesiana:    P = grado de creencia (más útil en IA)
```

---
## 2. Eventos y espacio muestral

In [ ]:
import random

# Simulación: lanzar un dado 10000 veces
np.random.seed(42)
N = 10000
lanzamientos = np.random.randint(1, 7, size=N)

print("=== Dado de 6 caras ===")
print(f"Espacio muestral: {set(lanzamientos)}")

# Frecuencias
for cara in range(1, 7):
    frec = np.sum(lanzamientos == cara) / N
    barra = "█" * int(frec * 100)
    print(f"  P(cara={cara}) ≈ {frec:.4f}  {barra}")

print(f"\nSuma total: {sum(np.sum(lanzamientos == c)/N for c in range(1,7)):.4f}  (debe ser 1)")

# Eventos compuestos
print("\n=== Eventos compuestos ===")
P_par  = np.sum(lanzamientos % 2 == 0) / N
P_mayor3 = np.sum(lanzamientos > 3) / N
P_par_y_mayor3 = np.sum((lanzamientos % 2 == 0) & (lanzamientos > 3)) / N
P_par_o_mayor3 = np.sum((lanzamientos % 2 == 0) | (lanzamientos > 3)) / N

print(f"P(par)          = {P_par:.4f}  (teórico: 0.5)")
print(f"P(>3)           = {P_mayor3:.4f}  (teórico: 0.5)")
print(f"P(par ∩ >3)     = {P_par_y_mayor3:.4f}  (teórico: {2/6:.4f})  → 4,6")
print(f"P(par ∪ >3)     = {P_par_o_mayor3:.4f}  (teórico: {4/6:.4f})  → 2,4,5,6")
print(f"P(par) + P(>3) - P(par∩>3) = {P_par + P_mayor3 - P_par_y_mayor3:.4f}  ← regla de adición")

# Convergencia a la probabilidad teórica
fig, ax = plt.subplots(figsize=(8, 3))
ns = np.arange(1, N+1)
frec_acum = np.cumsum(lanzamientos == 6) / ns
ax.plot(ns, frec_acum, lw=1, color='steelblue', label='Frecuencia observada')
ax.axhline(1/6, color='red', ls='--', label=f'P teórica = 1/6 = {1/6:.4f}')
ax.set_xscale('log'); ax.legend()
ax.set_xlabel("Número de lanzamientos"); ax.set_ylabel("P(cara=6)")
ax.set_title("Ley de los grandes números")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. Probabilidad condicional

```
P(A | B) = P(A ∩ B) / P(B)

"Probabilidad de A DADO QUE B ha ocurrido"

Ejemplo:
  P(cara > 4 | cara par) = ?

  P(>4 ∩ par) = P({6})         = 1/6
  P(par)       = P({2,4,6})    = 3/6 = 1/2

  P(>4 | par) = (1/6) / (1/2) = 1/3

Independencia: A y B son independientes si
  P(A | B) = P(A)   →   P(A ∩ B) = P(A) × P(B)
```

In [ ]:
# Ejemplo real: detección de spam
# P(spam) = 0.30 (30% de emails son spam)
# P(contiene "oferta" | spam) = 0.85
# P(contiene "oferta" | no spam) = 0.05

P_spam      = 0.30
P_no_spam   = 0.70
P_oferta_dado_spam    = 0.85
P_oferta_dado_nospam  = 0.05

P_oferta = P_oferta_dado_spam * P_spam + P_oferta_dado_nospam * P_no_spam
print(f"P(email contiene 'oferta') = {P_oferta:.4f}")

# Tabla de contingencia
N = 10000
n_spam    = int(N * P_spam)
n_nospam  = N - n_spam

tabla = {
    ("spam",    "oferta"):  int(n_spam   * P_oferta_dado_spam),
    ("spam",    "no oferta"): int(n_spam   * (1 - P_oferta_dado_spam)),
    ("no spam", "oferta"):  int(n_nospam * P_oferta_dado_nospam),
    ("no spam", "no oferta"): int(n_nospam * (1 - P_oferta_dado_nospam)),
}

print("\nTabla de contingencia (de 10000 emails):")
print(f"{'':15} {'con oferta':>12} {'sin oferta':>12} {'Total':>10}")
print(f"{'spam':15} {tabla[('spam','oferta')]:>12} {tabla[('spam','no oferta')]:>12} {n_spam:>10}")
print(f"{'no spam':15} {tabla[('no spam','oferta')]:>12} {tabla[('no spam','no oferta')]:>12} {n_nospam:>10}")
total_oferta = tabla[('spam','oferta')] + tabla[('no spam','oferta')]
print(f"{'Total':15} {total_oferta:>12} {N-total_oferta:>12} {N:>10}")

---
## 4. Teorema de Bayes

```
P(A | B) = P(B | A) × P(A) / P(B)

En lenguaje de IA:

P(hipótesis | datos) = P(datos | hipótesis) × P(hipótesis) / P(datos)
      ↑ posterior          ↑ verosimilitud      ↑ prior

Ejemplo: ¿Es spam dado que contiene "oferta"?

  P(spam | oferta) = P(oferta | spam) × P(spam) / P(oferta)
                   = 0.85 × 0.30 / 0.29
                   ≈ 0.879

Interpretación bayesiana:
  Prior:     P(spam) = 0.30  (creencia antes de ver el email)
  Evidencia: el email contiene "oferta"
  Posterior: P(spam|oferta) ≈ 0.88  (creencia actualizada)
```

In [ ]:
def bayes(P_B_dado_A, P_A, P_B):
    return P_B_dado_A * P_A / P_B

P_spam_dado_oferta = bayes(P_oferta_dado_spam, P_spam, P_oferta)
print(f"P(spam | contiene 'oferta') = {P_spam_dado_oferta:.4f} = {P_spam_dado_oferta*100:.1f}%")

# Actualización bayesiana secuencial
print("\n=== Actualización bayesiana secuencial ===")
print("Cada nueva evidencia actualiza la probabilidad:")

# Características del email y sus probabilidades condicionales
evidencias = [
    ("contiene 'oferta'",   0.85, 0.05),
    ("contiene 'gratis'",   0.70, 0.03),
    ("remitente desconocido",0.90, 0.20),
    ("sin asunto",          0.40, 0.10),
]

P_spam_actual = P_spam
print(f"  Prior: P(spam) = {P_spam_actual:.4f}")

for desc, P_ev_spam, P_ev_nospam in evidencias:
    P_ev = P_ev_spam * P_spam_actual + P_ev_nospam * (1 - P_spam_actual)
    P_spam_actual = bayes(P_ev_spam, P_spam_actual, P_ev)
    print(f"  + {desc:<35} → P(spam) = {P_spam_actual:.4f}")

print(f"\nDecisión: {'SPAM' if P_spam_actual > 0.5 else 'NO SPAM'} (umbral 0.5)")

---
## 5. Distribuciones de probabilidad

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 1. Uniforme discreta
ax = axes[0, 0]
caras = np.arange(1, 7)
ax.bar(caras, [1/6]*6, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_title("Uniforme discreta (dado)")
ax.set_xlabel("Cara"); ax.set_ylabel("P(X=k)")
ax.set_ylim(0, 0.25)

# 2. Binomial: n ensayos, P éxito = p
ax = axes[0, 1]
n, p = 20, 0.3
k = np.arange(0, n+1)
pmf = stats.binom.pmf(k, n, p)
ax.bar(k, pmf, color='salmon', alpha=0.7, edgecolor='black')
ax.set_title(f"Binomial (n={n}, p={p})")
ax.set_xlabel("k (éxitos)"); ax.set_ylabel("P(X=k)")
media_b = n * p
ax.axvline(media_b, color='red', ls='--', label=f"μ={media_b}")
ax.legend()

# 3. Poisson
ax = axes[0, 2]
lam = 4  # tasa promedio
k = np.arange(0, 15)
pmf_p = stats.poisson.pmf(k, lam)
ax.bar(k, pmf_p, color='mediumpurple', alpha=0.7, edgecolor='black')
ax.set_title(f"Poisson (λ={lam})")
ax.set_xlabel("k"); ax.set_ylabel("P(X=k)")
ax.axvline(lam, color='red', ls='--', label=f"λ={lam}")
ax.legend()

# 4. Normal
ax = axes[1, 0]
x = np.linspace(-4, 4, 300)
for mu, sigma, color, ls in [(0, 1, 'blue', '-'), (0, 2, 'red', '--'), (1, 1, 'green', ':')]:
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color=color, ls=ls,
            label=f"μ={mu}, σ={sigma}", lw=2)
ax.set_title("Normal (Gaussiana)")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend(); ax.grid(True, alpha=0.3)

# 5. Exponencial
ax = axes[1, 1]
x = np.linspace(0, 5, 300)
for rate, color in [(0.5, 'blue'), (1, 'red'), (2, 'green')]:
    ax.plot(x, stats.expon.pdf(x, scale=1/rate), color=color, lw=2, label=f"λ={rate}")
ax.set_title("Exponencial")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend(); ax.grid(True, alpha=0.3)

# 6. Regla 68-95-99.7 de la normal
ax = axes[1, 2]
x = np.linspace(-4, 4, 300)
ax.plot(x, stats.norm.pdf(x), 'black', lw=2)
for sigma, color, alpha_val in [(1, 'blue', 0.5), (2, 'green', 0.3), (3, 'red', 0.2)]:
    mask = np.abs(x) <= sigma
    ax.fill_between(x[mask], stats.norm.pdf(x[mask]), alpha=alpha_val, color=color,
                    label=f"±{sigma}σ: {stats.norm.cdf(sigma)-stats.norm.cdf(-sigma):.1%}")
ax.set_title("Regla 68-95-99.7")
ax.set_xlabel("σ"); ax.set_ylabel("f(x)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Distribuciones de probabilidad", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# scipy.stats: el toolkit de distribuciones
print("=== scipy.stats: normal estándar ===")
dist = stats.norm(loc=0, scale=1)  # μ=0, σ=1

print(f"PDF en x=0:    {dist.pdf(0):.4f}   (densidad, no probabilidad)")
print(f"CDF en x=1.96: {dist.cdf(1.96):.4f} (P(X ≤ 1.96))")
print(f"P(-1 ≤ X ≤ 1): {dist.cdf(1) - dist.cdf(-1):.4f}  (±1σ ≈ 68%)")
print(f"P(-2 ≤ X ≤ 2): {dist.cdf(2) - dist.cdf(-2):.4f}  (±2σ ≈ 95%)")
print(f"Percentil 95%: {dist.ppf(0.95):.4f}  (valor x tal que P(X≤x)=0.95)")

print("\n=== Muestrear de distribuciones ===")
muestras_normal = stats.norm.rvs(loc=170, scale=8, size=1000, random_state=42)
print(f"1000 alturas simuladas (μ=170cm, σ=8cm):")
print(f"  Media observada: {muestras_normal.mean():.2f}cm")
print(f"  Desv. observada: {muestras_normal.std():.2f}cm")
print(f"  Min: {muestras_normal.min():.1f}cm, Max: {muestras_normal.max():.1f}cm")

print("\n=== Binomial: probabilidades acumuladas ===")
# Si lanzamos 10 monedas, ¿cuál es la P de obtener 7 o más caras?
P_7_o_mas = 1 - stats.binom.cdf(6, n=10, p=0.5)
print(f"P(X ≥ 7) con n=10, p=0.5: {P_7_o_mas:.4f}")

# Verificar simulando
resultados = stats.binom.rvs(n=10, p=0.5, size=100000, random_state=42)
print(f"P(X ≥ 7) simulada:         {np.mean(resultados >= 7):.4f}")

---
## Práctica

### Ejercicio 1 — Simulación de Montecarlo

**Enunciado:** Estima el número π usando el método de Montecarlo:

```
Idea: inscribe un círculo de radio 1 en un cuadrado de lado 2.
Genera puntos aleatorios en el cuadrado.
La fracción de puntos dentro del círculo ≈ π/4.

Un punto (x,y) está dentro del círculo si x² + y² ≤ 1
```

1. Simula con 1000, 10000 y 100000 puntos. ¿Cómo mejora la estimación?
2. Visualiza los puntos dentro y fuera del círculo
3. Traza el error de estimación en función del número de puntos

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Estimaciones
for N in [1000, 10000, 100000]:
    x = np.random.uniform(-1, 1, N)
    y = np.random.uniform(-1, 1, N)
    dentro = (x**2 + y**2) <= 1
    pi_est = 4 * np.sum(dentro) / N
    print(f"N={N:>7}: π ≈ {pi_est:.6f}  (error: {abs(pi_est - np.pi):.6f})")

# Visualización
N_vis = 5000
x = np.random.uniform(-1, 1, N_vis)
y = np.random.uniform(-1, 1, N_vis)
dentro = (x**2 + y**2) <= 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter
axes[0].scatter(x[dentro], y[dentro], s=1, c='steelblue', alpha=0.5, label="dentro")
axes[0].scatter(x[~dentro], y[~dentro], s=1, c='salmon', alpha=0.5, label="fuera")
theta = np.linspace(0, 2*np.pi, 300)
axes[0].plot(np.cos(theta), np.sin(theta), 'black', lw=2)
axes[0].set_aspect('equal'); axes[0].set_title(f"Montecarlo: π ≈ {4*dentro.sum()/N_vis:.4f}")
axes[0].legend(markerscale=10)

# Convergencia
ns = np.arange(100, 50001, 100)
x_all = np.random.uniform(-1, 1, 50000)
y_all = np.random.uniform(-1, 1, 50000)
d_all = (x_all**2 + y_all**2) <= 1
pi_acum = [4 * d_all[:n].sum() / n for n in ns]
axes[1].plot(ns, pi_acum, lw=1, color='steelblue', label='Estimación')
axes[1].axhline(np.pi, color='red', ls='--', label=f'π real = {np.pi:.5f}')
axes[1].set_xlabel("N puntos"); axes[1].set_ylabel("Estimación de π")
axes[1].set_title("Convergencia"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 2 — Filtro Bayesiano

**Enunciado:** Implementa un clasificador Bayesiano Naive para detectar correos de phishing.

```
Características del email:
  P(phishing) = 0.05  (el 5% de emails son phishing)

  P(característica | phishing)  P(característica | legítimo)
  'urgente':      0.70               0.03
  'contraseña':   0.60               0.02
  'haz clic':     0.80               0.10
  'banco':        0.50               0.05
  'cuenta bloqueada': 0.65           0.01
```

Dados estos tres emails, ¿cuál es la probabilidad de phishing?
```
Email A: contiene 'urgente', 'haz clic'
Email B: contiene 'banco', 'contraseña', 'cuenta bloqueada', 'urgente'
Email C: no contiene ninguna de las palabras clave
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np

# Probabilidades condicionales
caracteristicas = {
    'urgente':          (0.70, 0.03),
    'contraseña':       (0.60, 0.02),
    'haz clic':         (0.80, 0.10),
    'banco':            (0.50, 0.05),
    'cuenta bloqueada': (0.65, 0.01),
}

def clasificar(palabras_presentes, P_phish_prior=0.05):
    log_P_phish   = np.log(P_phish_prior)
    log_P_legit   = np.log(1 - P_phish_prior)

    for palabra, (p_phish, p_legit) in caracteristicas.items():
        if palabra in palabras_presentes:
            log_P_phish += np.log(p_phish)
            log_P_legit += np.log(p_legit)
        else:
            log_P_phish += np.log(1 - p_phish)
            log_P_legit += np.log(1 - p_legit)

    # Convertir log-probabilidades a probabilidades
    max_log = max(log_P_phish, log_P_legit)
    p1 = np.exp(log_P_phish - max_log)
    p2 = np.exp(log_P_legit - max_log)
    P_phish_post = p1 / (p1 + p2)
    return P_phish_post

emails = {
    "Email A": {'urgente', 'haz clic'},
    "Email B": {'banco', 'contraseña', 'cuenta bloqueada', 'urgente'},
    "Email C": set(),
}

for nombre, palabras in emails.items():
    p = clasificar(palabras)
    decision = "⚠️  PHISHING" if p > 0.5 else "✓  Legítimo"
    print(f"{nombre}: P(phishing) = {p:.4f}  → {decision}")
    if palabras:
        print(f"  Palabras: {palabras}")
```
</details>

---
### Ejercicio 3 — Distribución normal y percentiles

**Enunciado:** Las puntuaciones de un test de aptitud siguen una distribución normal con μ=100, σ=15.

1. ¿Qué porcentaje obtiene más de 130 puntos?
2. ¿Qué porcentaje está entre 85 y 115?
3. ¿Cuál es la puntuación mínima para estar en el top 5%?
4. Simula 1000 estudiantes y visualiza el histograma junto con la PDF teórica
5. ¿Cuántos estudiantes de 1000 superarán 145 puntos (3σ)?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

mu, sigma = 100, 15
dist = stats.norm(loc=mu, scale=sigma)

# 1.
p_mas_130 = 1 - dist.cdf(130)
print(f"P(X > 130) = {p_mas_130:.4f} = {p_mas_130*100:.2f}%")

# 2.
p_85_115 = dist.cdf(115) - dist.cdf(85)
print(f"P(85 ≤ X ≤ 115) = {p_85_115:.4f} = {p_85_115*100:.2f}%")

# 3.
top5 = dist.ppf(0.95)
print(f"Top 5%: puntuación mínima = {top5:.1f}")

# 4. Simulación + visualización
muestras = dist.rvs(size=1000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(muestras, bins=30, density=True, alpha=0.6, color='steelblue', label='Simulados (n=1000)')
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
ax.plot(x, dist.pdf(x), 'r-', lw=2, label='PDF teórica')
ax.axvline(top5, color='orange', ls='--', label=f'Top 5% ({top5:.1f})')
ax.axvline(130, color='purple', ls='--', label='130 puntos')
ax.set_xlabel('Puntuación'); ax.set_ylabel('Densidad')
ax.set_title('Distribución de puntuaciones (μ=100, σ=15)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 5.
p_145 = 1 - dist.cdf(145)
esperados = p_145 * 1000
observados = np.sum(muestras > 145)
print(f"\nP(X > 145) = {p_145:.5f}")
print(f"Esperados en 1000: {esperados:.2f}")
print(f"Observados en simulación: {observados}")
```
</details>

---
### Ejercicio 4 — Teorema Central del Límite

**Enunciado:** El Teorema Central del Límite (TCL) establece que la media de muchas muestras independientes tiende a distribución normal, sin importar la distribución original.

Demuéstralo empíricamente:
1. Genera 10000 muestras de tamaño `n` de una distribución uniforme en [0,1]
2. Calcula la media de cada muestra
3. Visualiza el histograma de las medias para n=1, 5, 30, 100
4. ¿Para qué n la distribución de medias ya parece normal?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
N_muestras = 10000
ns = [1, 5, 30, 100]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, n in zip(axes, ns):
    # Generar N_muestras muestras de tamaño n
    datos   = np.random.uniform(0, 1, size=(N_muestras, n))
    medias  = datos.mean(axis=1)

    ax.hist(medias, bins=50, density=True, alpha=0.7, color='steelblue')

    # Curva normal teórica (TCL: μ=0.5, σ=1/(2√3·√n))
    mu_teo    = 0.5
    sigma_teo = (1/12)**0.5 / np.sqrt(n)  # σ_original/√n, σ_uniforme = 1/√12
    x = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu_teo, sigma_teo), 'r-', lw=2)

    # Test de normalidad
    _, p_valor = stats.shapiro(medias[:500])  # Shapiro con submuestra
    normal = "Normal ✓" if p_valor > 0.05 else f"No normal (p={p_valor:.3f})"

    ax.set_title(f"n = {n}\n{normal}", fontsize=9)
    ax.set_xlabel("Media muestral")
    if ax == axes[0]: ax.set_ylabel("Densidad")

plt.suptitle("Teorema Central del Límite (distribución uniforme → media)", fontsize=11)
plt.tight_layout()
plt.show()
print("→ A partir de n≈30, la distribución de medias ya es prácticamente normal")
```
</details>

---
### Ejercicio 5 (reto) — Generador de datos sintéticos

**Enunciado:** Crea un generador de datasets sintéticos para clasificación binaria:

```
Clase 0 (clientes que no compran):
  - edad:     Normal(μ=45, σ=12)
  - ingresos: Normal(μ=28000, σ=8000)
  - visitas:  Poisson(λ=2)

Clase 1 (clientes que compran):
  - edad:     Normal(μ=35, σ=10)
  - ingresos: Normal(μ=45000, σ=12000)
  - visitas:  Poisson(λ=8)
```

1. Genera 500 clientes de cada clase
2. Visualiza las distribuciones de cada característica por clase
3. ¿Qué característica separa mejor las dos clases?
4. Implementa un clasificador Bayesiano simple basado en estas distribuciones

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
N = 500

# Generar datos
clase0 = np.column_stack([
    stats.norm.rvs(45, 12, N), stats.norm.rvs(28000, 8000, N), stats.poisson.rvs(2, size=N)
])
clase1 = np.column_stack([
    stats.norm.rvs(35, 10, N), stats.norm.rvs(45000, 12000, N), stats.poisson.rvs(8, size=N)
])

nombres = ["Edad", "Ingresos", "Visitas"]

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (ax, nombre) in enumerate(zip(axes, nombres)):
    ax.hist(clase0[:, i], bins=30, alpha=0.6, density=True, color='salmon', label='No compra')
    ax.hist(clase1[:, i], bins=30, alpha=0.6, density=True, color='steelblue', label='Compra')
    ax.set_title(nombre); ax.legend()
plt.tight_layout(); plt.show()

# Clasificador Bayesiano
def log_verosimilitud_normal(x, mu, sigma):
    return stats.norm.logpdf(x, mu, sigma)

def clasificar(edad, ingresos, visitas, P_compra=0.5):
    log_p0  = np.log(1 - P_compra)
    log_p0 += log_verosimilitud_normal(edad, 45, 12)
    log_p0 += log_verosimilitud_normal(ingresos, 28000, 8000)
    log_p0 += stats.poisson.logpmf(int(visitas), 2)

    log_p1  = np.log(P_compra)
    log_p1 += log_verosimilitud_normal(edad, 35, 10)
    log_p1 += log_verosimilitud_normal(ingresos, 45000, 12000)
    log_p1 += stats.poisson.logpmf(int(visitas), 8)

    return 1 if log_p1 > log_p0 else 0

# Evaluar en datos de test
X = np.vstack([clase0, clase1])
y = np.array([0]*N + [1]*N)
pred = [clasificar(*fila) for fila in X]
acc = np.mean(np.array(pred) == y)
print(f"Accuracy del clasificador Bayesiano: {acc:.4f} = {acc*100:.1f}%")
```
</details>

---
## Resumen

```python
from scipy import stats

# Distribuciones
dist_normal = stats.norm(loc=0, scale=1)
dist_normal.pdf(0)        # densidad
dist_normal.cdf(1.96)     # P(X ≤ 1.96)
dist_normal.ppf(0.95)     # percentil 95
dist_normal.rvs(size=100) # muestras aleatorias

# Bayes
# P(A|B) = P(B|A) × P(A) / P(B)
P_posterior = P_verosimilitud * P_prior / P_evidencia
```

## ¿Qué viene en la próxima clase?
En la **Clase 4** medimos los datos que ya tenemos: estadística descriptiva con media, mediana, desviación estándar y correlación.